In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from copy import deepcopy
from gen_variable_standard_static import find_aggregate_variable_names_gen_mod, \
find_all_variable_names_gen_mod, check_variable_data_exists, sources_checker, \
check_variable_data_exists_single_system, metrics_search_for_fragment_df, \
metrics_search_for_two_fragments_df
from tqdm import tqdm

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records'],
      dtype='str')

In [4]:
systems_parquet_pow = systems_cleaned[
    systems_cleaned['is_lake_parquet_data']
    & systems_cleaned['has_power_data']
]
systems_parquet_pow_ids = set(systems_parquet_pow.system_id)
len(systems_parquet_pow_ids)

107

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [6]:
systems_parquet_pow_ids.difference(metrics_id_set)

set()

Hence, no parquet-power systems  in systems_cleaned outside metric_id

In [8]:
pow_metrics = metrics_search_for_fragment_df(metrics_df, 'pow')
pow_metrics_set = set(pow_metrics.system_id)
len(pow_metrics_set)

110

In [9]:
pow_metrics_set.difference(systems_parquet_pow_ids)

{1251, 1327, 1341}

## 1251 investigation

In [17]:
system_id = 1251
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
347,1251,3375,ac_current,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current__3375
348,1251,3373,ac_power_kW,AC power,kW,W,1000.0,0.0,,avg,NaN,NaN,,ac_power_kw__3373
349,1251,3374,ac_voltage,AC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,ac_voltage__3374
350,1251,3372,energy,AC energy,-,-,1.0,0.0,,NaN,NaN,NaN,,energy__3372


What's going on?  One possibility

In [11]:
systems_dir = Path("../../../data/raw/parquet-systems/")
systems_pq = pq.ParquetDataset(systems_dir)
systems_df = systems_pq.read().to_pandas()


In [12]:
systems_df[systems_df['system_id'] == 1251]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id


What's going on with 'site_id'?

In [13]:
systems_df[systems_df['site_id'] == 1251]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id
52,28.953,1608 Tennessee District E,,4.14,Residential - NOLA -20 - District E,1251,2018-03-31 17:23:10,1263


In [14]:
systems_cleaned[systems_cleaned['system_id'] == 1263]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records
52,1263,Residential - NOLA -20 - District E,"New Orleans, LA",America/Chicago,29.9809,-90.0023,10.0,4.14,Cfa,44,...,False,True,True,True,True,False,multi-Si,multicrystalline_Si,PVDAQ General,1239


Given the matches in power_maximum and public_name, these are almost certainly the same.

In [16]:
system_id = 1263
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
395,1263,3423,A_avg,AC current other,A,A,1.000,0.0,,avg,NaN,NaN,,a_avg__3423
396,1263,3422,PPV_avg,AC voltage,V,V,1.000,0.0,,avg,NaN,NaN,,ppv_avg__3422
397,1263,3421,W_avg,AC power,W,W,1.000,0.0,,avg,NaN,NaN,,w_avg__3421
398,1263,3420,Wh_sum,AC energy,kWh,kWh,0.001,0.0,,sum,NaN,NaN,,wh_sum__3420


One possible answer -- really misaligned data for System 1263

#### Check Missingness

In [28]:
path_1251 = Path('../../../../data_ds_project/systems/parquet/1251/')
my_power_id = 3373
pq_1251 = pq.ParquetDataset(
    path_1251,
    filters = [
        ('metric_id', '==', my_power_id)
    ]
)
df_1251 = pq_1251.read().to_pandas()
df_1251.tail()

FileNotFoundError: ../../../../data_ds_project/systems/parquet/1251

In [30]:
path_1263 = Path('../../../../data_ds_project/systems/parquet/1263/')
my_power_id = 3421
pq_1263 = pq.ParquetDataset(
    path_1263,
    filters = [
        ('metric_id', '==', my_power_id)
    ]
)
df_1263 = pq_1263.read().to_pandas()
df_1263.tail()

,measured_on,utc_measured_on,metric_id,value
355216,2015-09-21 17:25:00,NaT,3421,217.0
355217,2015-09-21 17:30:00,NaT,3421,145.0
355218,2015-09-21 17:35:00,NaT,3421,89.0
355219,2015-09-21 18:00:00,NaT,3421,-6.0
355220,2015-09-21 18:05:00,NaT,3421,-6.0


AHA -- did not download System 1251, since not in systems_cleaned.  Can ignore unless there is a problem.

Also, appears to be nothing in the online database for System 1251, so really nothing here!

### System 1327 vs. Site 1327 / System 1346

In [27]:
system_id = 1327
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_rows_metrics

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
846,1327,287,VATotal,AC other,VA,VA,1.0,0.0,,avg,NaN,NaN,,vatotal__287
847,1327,286,RTW,AC power,W,W,1.0,0.0,,avg,NaN,NaN,,rtw__286
848,1327,288,WHTotal,AC energy,Wh,Wh,1.0,0.0,,avg,NaN,NaN,,whtotal__288


In [21]:
systems_df[systems_df['system_id'] == 1327]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id


In [22]:
systems_df[systems_df['site_id'] == 1327]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id
4,45.6596,6015 Prytania St. District A South. DISABLED. ...,,8.82,Residential - NOLA -13 - Net Meter - District ...,1327,2012-03-09 18:03:00,1346


In [19]:
systems_cleaned[systems_cleaned['system_id'] == 1346]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records
120,1346,Residential - NOLA -13 - Net Meter - District ...,"New Orleans, LA",America/Chicago,29.9288,-90.1273,10.0,8.82,Cfa,44,...,False,False,True,False,True,False,mono-Si,monocrystalline_Si,PVDAQ General,957


In [24]:
metrics_df[metrics_df['system_id'] == 1346]

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
920,1346,3519,A_avg,AC current other,A,A,1.000,0.0,,avg,NaN,NaN,,a_avg__3519
921,1346,3517,W_n_avg,AC other,W,W,1.000,0.0,,avg,NaN,NaN,,w_n_avg__3517
922,1346,3516,Wh_n_sum,AC other,kWh,kWh,0.001,0.0,,NaN,NaN,NaN,,wh_n_sum__3516


Almost surely the same mixup, but not a 1:1 matchup of variables, either

#### check missingness

### System 1341 vs. Site 1341 / System 1360

In [25]:
systems_df[systems_df['system_id'] == 1341]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id


In [20]:
systems_df[systems_df['site_id'] == 1341]

,area,comments,ended_on,power,public_name,site_id,started_on,system_id
113,22.519,5018 N. Prieur District E,,2.8,Residential - NOLA -27 - Net Meter - District E,1341,2012-02-16 22:43:00,1360


In [23]:
systems_cleaned[systems_cleaned['system_id'] == 1360]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_temperature_data,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records
134,1360,Residential - NOLA -27 - Net Meter - District E,"New Orleans, LA",America/Chicago,29.9809,-90.0023,10.0,2.8,Cfa,44,...,False,False,True,False,True,False,multi-Si,multicrystalline_Si,PVDAQ General,1253


In [26]:
metrics_df[metrics_df['system_id']==1360]

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
962,1360,3575,A_avg,AC current other,A,A,1.000,0.0,,avg,NaN,NaN,,a_avg__3575
963,1360,3573,W_n_avg,AC other,W,W,1.000,0.0,,avg,NaN,NaN,,w_n_avg__3573
964,1360,3572,Wh_n_sum,AC other,kWh,kWh,0.001,0.0,,NaN,NaN,NaN,,wh_n_sum__3572


## Conclusion:

The 'bad' system_id's are almost-surely misplaced site_id's.  (All for various NOLA districts.)
But the true system_id's corresponding also have corresponding metadata.
Unless there's a bad case of missingness, we should not worry about dropping the double-ups.

#### Some quick checks on AC power counts (is 102 correct?)

In [31]:
ac_pow_metrics = metrics_search_for_two_fragments_df(metrics_df, 'pow', 'ac', 'and')
ac_pow_metrics_set = set(ac_pow_metrics.system_id)
ac_pow_with_systems = ac_pow_metrics_set.intersection(systems_parquet_pow_ids)
len(ac_pow_with_systems)

102

Yes.  Continue!